In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/accounts.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/alerts.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/transactions.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/paramFiles/alertPatterns.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/paramFiles/degree.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/paramFiles/accounts.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/paramFiles/transactionType.csv
/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/stat/cc.png
/kaggle/input/datasets/mo

In [49]:
df=pd.read_csv('/kaggle/input/datasets/mohammadhafees/ibm-amlsim-100k-accounts-10m-transactions/100Kvertices-10Medges/transactions.csv', nrows=100_000)
df.head()

,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID
0,1,5942,92982,TRANSFER,516.95,0,False,-1
1,2,86700,43995,TRANSFER,197.88,0,False,-1
2,3,86700,95516,TRANSFER,197.88,0,False,-1
3,4,86700,83911,TRANSFER,197.88,0,False,-1
4,5,86700,82801,TRANSFER,197.88,0,False,-1


In [50]:
df.drop(columns = ['TX_ID','TX_TYPE','ALERT_ID'],inplace=True)
df.head()


,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_AMOUNT,TIMESTAMP,IS_FRAUD
0,5942,92982,516.95,0,False
1,86700,43995,197.88,0,False
2,86700,95516,197.88,0,False
3,86700,83911,197.88,0,False
4,86700,82801,197.88,0,False


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   SENDER_ACCOUNT_ID    100000 non-null  int64  
 1   RECEIVER_ACCOUNT_ID  100000 non-null  int64  
 2   TX_AMOUNT            100000 non-null  float64
 3   TIMESTAMP            100000 non-null  int64  
 4   IS_FRAUD             100000 non-null  bool   
dtypes: bool(1), float64(1), int64(3)
memory usage: 3.1 MB


In [52]:
X = df[['SENDER_ACCOUNT_ID','RECEIVER_ACCOUNT_ID','TX_AMOUNT','TIMESTAMP']].copy()
y = df['IS_FRAUD'].astype(int)


In [53]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [54]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

In [55]:

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)


{0: np.float64(0.5007636645884974), 1: np.float64(327.8688524590164)}


In [56]:
import tensorflow 
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [57]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential([
    Input(shape=(4,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])



In [58]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

In [59]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [60]:
history = model.fit(X_train_trf,y_train,batch_size=50,epochs=100,verbose=1,validation_split=0.2,class_weight=class_weights)

Epoch 1/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8993 - loss: 0.6632 - val_accuracy: 0.7089 - val_loss: 0.4316
Epoch 2/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6775 - loss: 0.4863 - val_accuracy: 0.6653 - val_loss: 0.4723
Epoch 3/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.6774 - loss: 0.4258 - val_accuracy: 0.6697 - val_loss: 0.4794
Epoch 4/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6782 - loss: 0.3999 - val_accuracy: 0.6797 - val_loss: 0.4741
Epoch 5/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6891 - loss: 0.3889 - val_accuracy: 0.6705 - val_loss: 0.5164
Epoch 6/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6846 - loss: 0.3873 - val_accuracy: 0.6764 - val_loss: 0.4986
Epoch 7/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6811 - loss: 0.3826 - val_accuracy: 0.6832 - val_loss: 0.4655
Epoch 8/100
1280/1280 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6866 - loss: 0

In [61]:
y_prob = model.predict(X_test_trf)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [62]:
y_prob


array([[1.2381811e-27],
       [1.3808796e-09],
       [8.0995232e-01],
       ...,
       [1.1799178e-12],
       [1.6298140e-08],
       [7.8222585e-01]], dtype=float32)

In [63]:
y_pred = (y_prob >= 0.5).astype(int)
y_pred

array([[0],
       [0],
       [1],
       ...,
       [0],
       [0],
       [1]])

In [64]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy score:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



Accuracy score: 0.72475
              precision    recall  f1-score   support

           0       1.00      0.72      0.84     19978
           1       0.00      1.00      0.01        22

    accuracy                           0.72     20000
   macro avg       0.50      0.86      0.42     20000
weighted avg       1.00      0.72      0.84     20000

[[14473  5505]
 [    0    22]]
